# Lab 02: A pre-norm transformer block, and counting its parameters

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/hokwangchoi/hc-model-study/blob/master/labs/notebooks/lab02_transformer_block.ipynb) · lab page: [`labs/lab02.html`](../lab02.html) · file: [`labs/lab02_transformer_block.py`](../lab02_transformer_block.py)

**Time:** 60 min

## Goal

Assemble one decoder block the way GPT-2 (LayerNorm + GELU MLP) and Llama
(RMSNorm + SwiGLU) build it, then write a closed-form parameter count and
check it against PyTorch's own count. Lab 01 built attention; here it
becomes one sub-layer of a block.

## The maths, step by step

```text
LayerNorm, per token, over the last dim (d values):
    mu      = mean(x)                                   [B, N, 1]
    var     = mean((x - mu)^2)       (divide by d, NOT d - 1)
    LN(x)   = gamma * (x - mu) / sqrt(var + eps) + beta     gamma, beta: [d]

RMSNorm (Llama, Mistral, Qwen...): no mean subtraction, no beta:
    RMS(x)  = gamma * x / sqrt(mean(x^2) + eps)              gamma: [d]

Feed-forward network (FFN), applied to every token independently:
    GELU MLP:  W2 GELU(W1 x + b1) + b2            d -> d_ff -> d   (d_ff = 4d in GPT-2)
    SwiGLU:    W_down( SiLU(W_gate x) * (W_up x) )  d -> d_ff (twice) -> d
               SiLU(z) = z * sigmoid(z); the "gate" decides, per hidden unit,
               how much of W_up x passes through.

Self-attention: lab 01's multi-head attention (q/k/v/o projections).

Pre-norm block (every modern LLM and ViT):
    h = x + Attn(Norm1(x))          the residual stream x is never normalised
    y = h + FFN(Norm2(h))           itself; each sub-layer reads a normalised
                                    copy and ADDS its result back.
(The 2017 paper used post-norm, x = Norm(x + Attn(x)), which trains worse
when deep.)

Parameter count (d model width, f = d_ff):
    attention   4 d^2           + 4 d   biases (q, k, v, o), if bias
    GELU MLP    2 d f           + f + d biases, if bias
    SwiGLU      3 d f           (no biases in Llama-style models)
    LayerNorm   2 d  (gamma, beta)       RMSNorm  d  (gamma)
The number of heads does not change the count: heads only split d.
```

## Read alongside

- [Transformer, step by step](../../transformer/transformer_step_by_step.html): (the step-by-step page)
- [Transformer guide #architecture](../../transformer/transformer_guide.html#architecture): the block, expandable parts: #blk-ln (LayerNorm), #blk-residual, #blk-ffn
- [Transformer guide #modern](../../transformer/transformer_guide.html#modern): RMSNorm, SwiGLU, pre-norm
*Preview (covered in the next tutorial, Build an LLM)*
- [Build an LLM #model](../../build-llm/build_llm_guide.html#model): the same block in a trained LLM

## How to work

1. Run the setup cell and the imports cell.
2. Implement the functions in order, replacing `raise NotImplementedError  # TODO`. The hints above each cell open one at a time; try Hint 1 first.
3. Run each cell after editing it, then run the check cell at the end: `check.run_lab(2, namespace=globals())` checks the functions defined in this notebook (TODO = not started, PASS = matches the reference, FAIL = read the message).

Your work stays in this notebook; it does not change `labs/lab02_transformer_block.py` or the lab page. The reference solution is `labs/solutions/lab02_transformer_block.py`.

In [ ]:
# Setup: make the checker (labs/check.py) importable. Run this cell first.
# In a clone of the repository (Jupyter, VS Code) it uses labs/check.py;
# elsewhere (e.g. Google Colab) it downloads check.py, which needs only torch.
import pathlib
import sys
import urllib.request


def _find_labs_dir():
    here = pathlib.Path.cwd().resolve()
    for d in [here, *here.parents]:
        for cand in (d, d / "labs"):
            if (cand / "check.py").is_file() and (cand / "solutions").is_dir():
                return cand
    return None


LABS_DIR = _find_labs_dir()
if LABS_DIR is None:
    LABS_DIR = pathlib.Path("hc_model_study_labs").resolve()
    LABS_DIR.mkdir(exist_ok=True)
    if not (LABS_DIR / "check.py").is_file():
        urllib.request.urlretrieve("https://raw.githubusercontent.com/hokwangchoi/hc-model-study/master/labs/check.py", LABS_DIR / "check.py")
sys.path.insert(0, str(LABS_DIR))
import check  # noqa: E402

import torch  # noqa: E402
print("checker:", LABS_DIR / "check.py", "| torch", torch.__version__)

### Imports

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F

## Step 1: LayerNorm and RMSNorm

**Hints for `LayerNorm.forward`** (open one at a time):

<details><summary>Hint 1</summary>

```text
mean and variance over dim=-1 with keepdim=True, so they broadcast.
```

</details>

<details><summary>Hint 2</summary>

```text
LayerNorm uses the biased variance: x.var(dim=-1, unbiased=False),
or ((x - mu) ** 2).mean(-1, keepdim=True). Avoid E[x^2] - E[x]^2:
it loses precision in float32 when the mean is large.
```

</details>

<details><summary>Hint 3</summary>

```text
return self.weight * (x - mu) / torch.sqrt(var + self.eps) + self.bias
```

</details>

In [ ]:
class LayerNorm(nn.Module):
    """LayerNorm over the last dimension, written out (do not call F.layer_norm)."""

    def __init__(self, d: int, eps: float = 1e-5):
        super().__init__()
        self.eps = eps
        self.weight = nn.Parameter(torch.ones(d))   # gamma
        self.bias = nn.Parameter(torch.zeros(d))    # beta

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        """x: [..., d] -> [..., d]"""
        raise NotImplementedError  # TODO

**Hints for `RMSNorm.forward`** (open one at a time):

<details><summary>Hint 1</summary>

```text
one line once you have mean(x^2) over the last dim (keepdim=True).
```

</details>

<details><summary>Hint 2</summary>

```text
torch.rsqrt(t) = 1 / sqrt(t).
```

</details>

<details><summary>Hint 3</summary>

```text
return self.weight * x * torch.rsqrt(x.pow(2).mean(-1, keepdim=True) + self.eps)
```

</details>

In [ ]:
class RMSNorm(nn.Module):
    """RMSNorm over the last dimension: gamma * x / sqrt(mean(x^2) + eps)."""

    def __init__(self, d: int, eps: float = 1e-6):
        super().__init__()
        self.eps = eps
        self.weight = nn.Parameter(torch.ones(d))   # gamma

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        """x: [..., d] -> [..., d]"""
        raise NotImplementedError  # TODO

## Step 2: the two feed-forward networks

**Hints for `GeluMLP.forward`** (open one at a time):

<details><summary>Hint 1</summary>

```text
use F.gelu with its default (exact, erf-based) form;
approximate="tanh" differs by about 1e-3.
```

</details>

<details><summary>Hint 2</summary>

```text
return self.fc2(F.gelu(self.fc1(x)))
```

</details>

In [ ]:
class GeluMLP(nn.Module):
    """GPT-2 / ViT FFN: fc2(GELU(fc1(x))), d -> d_ff -> d."""

    def __init__(self, d: int, d_ff: int, bias: bool = True):
        super().__init__()
        self.fc1 = nn.Linear(d, d_ff, bias=bias)
        self.fc2 = nn.Linear(d_ff, d, bias=bias)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        raise NotImplementedError  # TODO

**Hints for `SwiGLU.forward`** (open one at a time):

<details><summary>Hint 1</summary>

```text
SiLU is applied to the gate branch only; the up branch stays linear.
```

</details>

<details><summary>Hint 2</summary>

```text
return self.w_down(F.silu(self.w_gate(x)) * self.w_up(x))
```

</details>

In [ ]:
class SwiGLU(nn.Module):
    """Llama FFN: w_down( SiLU(w_gate(x)) * w_up(x) ), d -> d_ff -> d."""

    def __init__(self, d: int, d_ff: int, bias: bool = False):
        super().__init__()
        self.w_gate = nn.Linear(d, d_ff, bias=bias)
        self.w_up = nn.Linear(d, d_ff, bias=bias)
        self.w_down = nn.Linear(d_ff, d, bias=bias)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        raise NotImplementedError  # TODO

## Step 3: self-attention (lab 01 again, as a sub-layer)

Given: `attend(q, k, v, causal)` is already written. Run the cell.

In [ ]:
def attend(q: torch.Tensor, k: torch.Tensor, v: torch.Tensor, causal: bool) -> torch.Tensor:
    """Given: lab 01's scaled_dot_product_attention, as torch's fused kernel.
    q, k, v [B, h, N, d_k] -> [B, h, N, d_k]."""
    return F.scaled_dot_product_attention(q, k, v, is_causal=causal)

**Hints for `SelfAttention.forward`** (open one at a time):

<details><summary>Hint 1</summary>

```text
project, split into heads, attend, merge, project: lab 01's
MultiHeadAttention, with the given attend() for the middle step.
```

</details>

<details><summary>Hint 2</summary>

```text
split: t.view(B, N, h, d // h).transpose(1, 2)
merge: o.transpose(1, 2).reshape(B, N, d)
```

</details>

<details><summary>Hint 3</summary>

```text
def split(t): return t.view(B, N, h, d // h).transpose(1, 2)
q, k, v = split(self.q_proj(x)), split(self.k_proj(x)), split(self.v_proj(x))
o = attend(q, k, v, causal)
return self.o_proj(o.transpose(1, 2).reshape(B, N, d))
```

</details>

In [ ]:
class SelfAttention(nn.Module):
    """Multi-head self-attention. Same maths as lab 01's MultiHeadAttention."""

    def __init__(self, d: int, n_heads: int, bias: bool = True):
        super().__init__()
        assert d % n_heads == 0
        self.n_heads = n_heads
        self.q_proj = nn.Linear(d, d, bias=bias)
        self.k_proj = nn.Linear(d, d, bias=bias)
        self.v_proj = nn.Linear(d, d, bias=bias)
        self.o_proj = nn.Linear(d, d, bias=bias)

    def forward(self, x: torch.Tensor, causal: bool = True) -> torch.Tensor:
        """x: [B, N, d] -> [B, N, d]  (returns only the output, no weights)"""
        raise NotImplementedError  # TODO

## Step 4: the pre-norm block

**Hints for `Block.forward`** (open one at a time):

<details><summary>Hint 1</summary>

```text
two lines, one per sub-layer, each of the form x = x + sublayer(norm(x)).
```

</details>

<details><summary>Hint 2</summary>

```text
the norm goes INSIDE the residual branch (pre-norm), not after the add.
```

</details>

<details><summary>Hint 3</summary>

```text
x = x + self.attn(self.norm1(x), causal=causal); x = x + self.ffn(self.norm2(x))
```

</details>

In [ ]:
class Block(nn.Module):
    """Pre-norm transformer block.

    norm: "layernorm" (GPT-2, ViT) or "rmsnorm" (Llama)
    ffn:  "gelu" (GPT-2, ViT)      or "swiglu"  (Llama)
    bias: whether the Linear layers have biases (GPT-2: yes, Llama: no)
    """

    def __init__(self, d: int, n_heads: int, d_ff: int, norm: str = "layernorm",
                 ffn: str = "gelu", bias: bool = True):
        super().__init__()
        Norm = {"layernorm": LayerNorm, "rmsnorm": RMSNorm}[norm]
        self.norm1 = Norm(d)
        self.attn = SelfAttention(d, n_heads, bias=bias)
        self.norm2 = Norm(d)
        self.ffn = GeluMLP(d, d_ff, bias=bias) if ffn == "gelu" else SwiGLU(d, d_ff, bias=bias)

    def forward(self, x: torch.Tensor, causal: bool = True) -> torch.Tensor:
        """x: [B, N, d] -> [B, N, d]"""
        raise NotImplementedError  # TODO

## Step 5: count the parameters in closed form

**Hints for `count_block_params`** (open one at a time):

<details><summary>Hint 1</summary>

```text
add up the pieces listed in the module docstring: attention, FFN, 2 norms.
```

</details>

<details><summary>Hint 2</summary>

```text
a Linear(a, b) holds a*b weights plus b biases (if bias).
```

</details>

<details><summary>Hint 3</summary>

```text
attn = 4*d*d + (4*d if bias else 0)
ffn  = 2*d*d_ff + (d_ff + d if bias else 0)   for gelu
       3*d*d_ff + (2*d_ff + d if bias else 0) for swiglu
norms = 2 * (2*d if layernorm else d)
```

</details>

In [ ]:
def count_block_params(d: int, n_heads: int, d_ff: int, norm: str = "layernorm",
                       ffn: str = "gelu", bias: bool = True) -> int:
    """Number of parameters of Block(d, n_heads, d_ff, norm, ffn, bias), from a formula.

    Do not build the module and call .numel(); work it out from the shapes.
    Sanity anchors:
      GPT-2 small block: d=768, d_ff=3072, layernorm, gelu, bias  -> 7,087,872
      Llama-2-7B block:  d=4096, d_ff=11008, rmsnorm, swiglu, no bias -> 202,383,360
    """
    raise NotImplementedError  # TODO

## Check

Run this after each function you finish. Steps you have not started show TODO; finished steps show PASS. For a full traceback of a failing step: `check.run_lab(2, globals(), verbose=True)`.

In [ ]:
check.run_lab(2, namespace=globals());

## Scratch

Try your functions here: print shapes, call them on small inputs. (Functions you have not written yet raise `NotImplementedError`.)

In [ ]:
# Try your modules here (Run the cells above first).
x = torch.randn(2, 5, 16)
print(LayerNorm(16)(x).mean(-1)[0])   # ~0 per token
blk = Block(16, 4, 64)
print(blk(x).shape)
count_block_params(768, 12, 3072)     # GPT-2 small block: 7,087,872